In [2]:
import numpy as np
import pandas as pd

from SymbolicDSGE import DSGESolver, ModelParser, Shock
from SymbolicDSGE.monte_carlo import (
    MCPipeline,
    custom_transform,
    pandas_operation,
)
from SymbolicDSGE.monte_carlo.step_factories import (
    kde_step,
    postproc_step,
    reference_filter_step,
    regression_step,
    simulation_step,
    standardize_step,
    transform_step,
    ljung_box_test_step,
    wald_test_step,
)
import cProfile

In [3]:
model, kalman = ModelParser("../../MODELS/misspec_test/reference.yaml").get_all()

solver = DSGESolver(model, kalman)
compiled = solver.compile()

ss_seed = np.zeros(5, dtype=np.float64)
reference = solver.solve(compiled, ss_seed=ss_seed)

dgp_model, dgp_kalman = ModelParser("../../MODELS/misspec_test/misspec.yaml").get_all()
dgp_comp = DSGESolver(dgp_model, dgp_kalman).compile()
dgp_sol = DSGESolver(dgp_model, dgp_kalman)
dgp = dgp_sol.solve(dgp_comp, ss_seed=ss_seed)

In [4]:
T = 200
n_obs = len(reference.compiled.observable_names)


@custom_transform
def custom_standardize(sample, output) -> int:
    n, p = sample.shape
    for j in range(p):
        mean = 0.0
        for i in range(n):
            mean += sample[i, j]
        mean /= n

        var = 0.0
        for i in range(n):
            var += (sample[i, j] - mean) ** 2
        std = (var / n) ** 0.5

        for i in range(n):
            output[i, j] = (sample[i, j] - mean) / std

    return 0


@pandas_operation
def get_std_obs_mean(*, traces):
    stacked = traces["payload.custom_std"]
    return pd.DataFrame({"mean": stacked.mean(axis=(0, 1))})


pipeline = MCPipeline(
    per_rep_steps=[
        simulation_step(
            T=T,
            target="dgp",
            shocks={
                "g,z": Shock(dist="norm", multivar=True, seed=0),
                "r": Shock(dist="norm", seed=1),
            },
            observables=True,
        ),
        reference_filter_step(),
        transform_step(
            "custom_std",
            custom_standardize,
            source="filter",
            field="innov",
            output_shape=(T, n_obs),
        ),
        standardize_step(
            "builtin_std",
            source="filter",
            field="innov",
        ),
        wald_test_step(
            "std_innov_mean",
            source="filter",
            field="std_innov",
            burn_in=20,
            target=np.zeros(n_obs),
            kind="mean",
        ),
    ],
    postproc_steps=[
        postproc_step(
            "custom_postproc",
            get_std_obs_mean,
        ),
        kde_step(
            "builtin_kde",
            trace="payload.builtin_std",
            grid_points=100,
        ),
    ],
)

In [12]:
mc = pipeline.run(
    reference=reference,
    dgp=dgp,
    n_rep=10000,
    n_jobs=-1,
    verbosity=2,
)

MC run concluded successfully in 0.25s with 40529.48 it/s.
Per-step Report:

	datagen: 0 failures, 47156.51 worker it/s (0.21 worker-s), 40529.48 wall it/s.
	filter: 0 failures, 2908.28 worker it/s (3.44 worker-s), 40529.48 wall it/s.
	custom_std: 0 failures, 219709.70 worker it/s (0.05 worker-s), 40529.48 wall it/s.
	builtin_std: 0 failures, 172673.13 worker it/s (0.06 worker-s), 40529.48 wall it/s.
	std_innov_mean: 0 failures, 87742.76 worker it/s (0.11 worker-s), 40529.48 wall it/s.

Post-processing Report:

	custom_postproc: Succeeded in 0.0184s.
	builtin_kde: Succeeded in 9.1390s.


In [ ]:
t_summary = pd.DataFrame(
    {
        name: {
            "mean_statistic": res.mean_statistic,
            "mean_pval": res.mean_pval,
            "rejection_rate": res.rejection_rate,
            "ci_low": res.pval_confidence_interval()[0],
            "ci_high": res.pval_confidence_interval()[1],
        }
        for name, res in mc.test_summaries.items()
    }
).T
print(t_summary.round(3))

NameError: name 'mc' is not defined